梯度指向值变化最大的方向，与切线正交

下面学习一下python的数学语言：

In [204]:
%matplotlib inline
#魔法命令用于显示图像
import numpy as np
from matplotlib_inline import backend_inline
#导入内联后端用于图像显示
from d2l import torch as d2l
#导入d2l中的torch库

def g(x):
    return 3 * x **2 - 4 * x
#定义函数g(x)，并通过return进行函数具化

In [205]:
def numerical_lim(g, x, h):
    #定义该函数，其参数为g, x, h(步长)
    return (g(x + h)-g(x)) / h
    #具化函数内容

h = 0.1
for i in range(5):
    print(f'h={h:.5f}, numerical limit={numerical_lim(g, 1, h):.5f}')
    h *= 0.1
    #打印float型且长度为5的步长h,并同理打印导数极限

h=0.10000, numerical limit=2.30000
h=0.01000, numerical limit=2.03000
h=0.00100, numerical limit=2.00300
h=0.00010, numerical limit=2.00030
h=0.00001, numerical limit=2.00003


自动求导

In [206]:
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [207]:
x.requires_grad_(True) 
# == x = torch.arange(4.0, requires_grad=True)
x.grad
#设置张量x的属性requires_grad为True，表示需要对x进行梯度保存
#x.grad表示输出x的梯度，但因为没有进行反向传播计算梯度，所以不显示

In [208]:
y = 2 * torch.dot(x, x)
y, y.shape

(tensor(28., grad_fn=<MulBackward0>), torch.Size([]))

In [209]:
y.backward()
x.grad
#进行反向传播，计算y对x的梯度，并表示出来

tensor([ 0.,  4.,  8., 12.])

In [210]:
x.grad == 4 * x

tensor([True, True, True, True])

知识理解：
在正向传播中我们计算出每一层的结果（激活值），再通过反向传播和被不断进行的链式法则求导，再代入我们正向传播时的计算出的每一行的激活值，可以算出每一行的梯度大小，这也就是为什么梯度诞生于反向传播的过程中。事实上，在最小化损失函数的过程中，我们通过反向传播计算每一层梯度从而对产生的误差进行每一层的归责，从而根据梯度来调正对应的权重等参数

In [211]:
x.grad.zero_()
#清楚累计的梯度
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

In [212]:
#假设y不是标量的话
x.grad.zero_()
y = x * x
y

tensor([0., 1., 4., 9.], grad_fn=<MulBackward0>)

In [213]:
y.sum().backward()
x.grad
#我们计算的其实是批量中每一个样本的偏导数之和

tensor([0., 2., 4., 6.])

分离计算：
简单理解为某内层函数的常数化处理z=y*x,y=x*x,我们如果求z关于X的导数会变成二次式，但如果我们把x的值带入y，那么我们把y视为常数，那么就是一次函数求导

In [214]:
#具体操作：
x.grad.zero_()
y = x * x
u = y.detach()
#此步操作就是分离计算，抛掉变量y而储存在一个常量u中

z = u * x 

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

In [215]:
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

python控制流的梯度计算

In [216]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b 
    else:
        c = 100 * b
    return c

a = torch.randn(size=(),requires_grad=True)
#torch.randn(size=()): 生成一个标量（0维张量），值来自标准正态分布，且需要梯度
d = f(a)
d.backward()

In [218]:
a.grad == d / a

tensor(True)

我的思考和计算：  
其实我们展开来看看这个计算流的f(x)到底长什么样子，也顺便解释一下为啥有a.grad == d / a  
显然，如果这样写，就会发现，f(x)和x是成线性相关的，那么我们逐层分析一下最终的表达式求出那个线性函数的斜率：  
* f(a) = a × 2ᵏ⁺¹ × S(a)  
* k = ceil(log₂(500 / |a|))（循环次数）  
* S(a) = 1 if a > 0 else 100（符号相关的缩放因子  
  
即a.grad == 2ᵏ⁺¹ × S(a)  